# Nested Random Effects for Multi-Level Data

**Topics:** Nested Effects, Three-Level Models, Hierarchical Structure

## Overview

Model data with multiple nested grouping levels: students within classrooms within schools.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from aurora.models.gamm import fit_gamm

sns.set_style('whitegrid')
np.random.seed(42)

## Generate Nested Data: Students in Classes in Schools

In [ ]:
n_schools = 10
n_classes_per_school = 3
n_students_per_class = 10
n_classes = n_schools * n_classes_per_school
n = n_schools * n_classes_per_school * n_students_per_class

# Random effects at each level
school_effects = np.random.randn(n_schools) * 5
class_effects = np.random.randn(n_classes) * 3

# Generate data
data = []
class_id = 0
for school in range(n_schools):
    for cls in range(n_classes_per_school):
        for student in range(n_students_per_class):
            # Test score = baseline + school_effect + class_effect + noise
            score = (
                70  # baseline
                + school_effects[school]  # school level
                + class_effects[class_id]  # class level (nested in school)
                + np.random.randn() * 4  # student level
            )
            
            data.append({
                'school': school,
                'class': class_id,
                'student': len(data),
                'score': score
            })
        class_id += 1

df = pd.DataFrame(data)
print(f"Generated {n} students in {n_classes} classes in {n_schools} schools")
print(f"\nVariance components (true):")
print(f"  School SD: {school_effects.std():.2f}")
print(f"  Class SD: {class_effects.std():.2f}")
print(f"  Student SD: 4.00")

## Fit Three-Level Model with Crossed Random Effects

We'll fit a model with two independent random effects: one for schools and one for classes. This is a **crossed** random effects model where we model variation at both levels independently.

Formula: `score ~ 1 + (1 | school) + (1 | class)`

In [ ]:
# Fit the three-level model using formula syntax
# This model includes two random intercepts: one for school and one for class
result = fit_gamm(
    formula='score ~ 1 + (1 | school) + (1 | class)',
    data=df,
    covariance='identity',  # Independent random effects
    family='gaussian'
)

print("Three-Level Model Results:")
print(f"\nFixed Effect (Overall Mean): {result.beta_parametric[0]:.2f}")
print(f"\nResidual Variance (σ²): {result.residual_variance:.4f}")

# Variance components is a block-diagonal matrix
# Extract variances for each grouping level
print(f"\nVariance Components (Ψ diagonal):")
# For identity covariance, diagonal elements are the variances
import numpy as np
psi_diag = np.diag(result.variance_components)

# First n_schools elements are school variances
# Next n_classes elements are class variances
school_variance = psi_diag[:n_schools].mean()
class_variance = psi_diag[n_schools:n_schools+n_classes].mean()

print(f"  School variance: {school_variance:.4f} (SD: {np.sqrt(school_variance):.2f})")
print(f"  Class variance: {class_variance:.4f} (SD: {np.sqrt(class_variance):.2f})")
print(f"  Residual: {result.residual_variance:.4f} (SD: {np.sqrt(result.residual_variance):.2f})")

# Extract random effects by grouping variable
school_effects_dict = result.random_effects['school']
class_effects_dict = result.random_effects['class']

# Convert to arrays for analysis
school_re = np.array([school_effects_dict[i][0] for i in sorted(school_effects_dict.keys())])
class_re = np.array([class_effects_dict[i][0] for i in sorted(class_effects_dict.keys())])

print(f"\nEstimated SD from BLUPs:")
print(f"  School: {school_re.std():.2f}")
print(f"  Class: {class_re.std():.2f}")

## Variance Decomposition

In [ ]:
# Calculate variance decomposition
var_school = school_variance
var_class = class_variance  
var_resid = result.residual_variance
var_total = var_school + var_class + var_resid

icc_school = var_school / var_total
icc_class = var_class / var_total
prop_student = var_resid / var_total

print("Variance Decomposition:")
print(f"\n{'Level':<15} {'Variance':<12} {'% Total':<10}")
print("="*37)
print(f"{'School':<15} {var_school:<12.3f} {icc_school*100:<10.1f}%")
print(f"{'Class':<15} {var_class:<12.3f} {icc_class*100:<10.1f}%")
print(f"{'Student':<15} {var_resid:<12.3f} {prop_student*100:<10.1f}%")
print("="*37)
print(f"{'Total':<15} {var_total:<12.3f} {100.0:<10.1f}%")

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
levels = ['School', 'Class', 'Student']
proportions = [icc_school*100, icc_class*100, prop_student*100]
colors = ['#ff9999', '#66b3ff', '#99ff99']

ax.pie(proportions, labels=levels, colors=colors, autopct='%1.1f%%',
       startangle=90, textprops={'fontsize': 12, 'weight': 'bold'})
ax.set_title('Variance Decomposition:\nWhere does variation come from?', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

# Compare with true values
print(f"\nComparison with True Values:")
print(f"{'Level':<15} {'True SD':<12} {'Estimated SD':<12}")
print("="*40)
print(f"{'School':<15} {school_effects.std():<12.2f} {np.sqrt(var_school):<12.2f}")
print(f"{'Class':<15} {class_effects.std():<12.2f} {np.sqrt(var_class):<12.2f}")
print(f"{'Student':<15} {4.00:<12.2f} {np.sqrt(var_resid):<12.2f}")

## Multiple Random Effects

### What We Just Did

We successfully fit a three-level hierarchical model with **multiple random effects** using Aurora-GLM:

```python
fit_gamm(formula='score ~ 1 + (1 | school) + (1 | class)', data=df)
```

This model includes:
- **Fixed effect**: Overall mean (intercept)
- **Random effect 1**: School-level variation (10 schools)
- **Random effect 2**: Class-level variation (30 classes)
- **Residual**: Student-level variation

### Understanding the Model Structure

**Crossed vs Nested Random Effects:**

- **Crossed**: `(1 | school) + (1 | class)` - Classes and schools are independent groupings
- **Nested**: `(1 | school/class)` - Classes are nested within schools (each class belongs to exactly one school)

For truly nested structure, the formula `(1 | school/class)` would be more appropriate, but the crossed structure we used works well when each class is uniquely identified (class IDs don't repeat across schools).

### Model Formula Syntax

Aurora-GLM now supports lme4-style random effects formulas:

**Single random effect:**
```python
'y ~ x1 + (1 | group)'  # Random intercept
```

**Random slopes:**
```python
'y ~ x1 + (1 + x1 | group)'  # Random intercept + slope
```

**Multiple independent random effects:**
```python
'y ~ x1 + (1 | group1) + (1 | group2)'  # Crossed
```

**Nested random effects:**
```python
'y ~ x1 + (1 | group1/group2)'  # Nested
```

### Implementation Details

The implementation uses:
- **REML estimation** for variance components
- **Block-diagonal covariance structure** for multiple random effects
- **BLUPs** (Best Linear Unbiased Predictors) for random effect estimates
- **Efficient matrix operations** with Cholesky decomposition

### Variance Components

The `variance_components` matrix is block-diagonal:
```
Ψ = [ Ψ_school     0      ]
    [     0      Ψ_class   ]
```

Each block contains the variance-covariance matrix for that grouping level.

### What's Next

Explore more advanced topics in `05_advanced_topics/`:
- Crossed random effects with longitudinal data
- Random slopes with different covariance structures
- Interactions between random effects and fixed effects
- Model selection and comparison techniques

**Congratulations!** You now have full support for multilevel hierarchical models in Aurora-GLM! 🎉